In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_1_mtd_performance AS
WITH latest_full_month AS (
    -- Skip the latest partial month; use the second-to-last distinct month
    SELECT MAX(invoice_month) AS full_month
    FROM (
        SELECT DISTINCT DATE_TRUNC('MONTH', invoice_date) AS invoice_month
        FROM workspace.cube.fact_invoice
    )
    WHERE invoice_month < DATE_TRUNC('MONTH', (SELECT MAX(invoice_date) FROM workspace.cube.fact_invoice))
),
monthly_stats AS (
    SELECT
        ds.store_id, ds.store_name, ds.manager_name,
        DATE_TRUNC('MONTH', fi.invoice_date) AS invoice_month,
        SUM(fi.invoice_amount) AS revenue,
        COUNT(DISTINCT fo.order_id) AS completed_orders
    FROM workspace.cube.fact_invoice fi
    JOIN workspace.cube.fact_order fo ON fi.order_id = fo.order_id
    JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
    CROSS JOIN latest_full_month lfm
    WHERE fo.order_status = 'COMPLETED'
      AND DATE_TRUNC('MONTH', fi.invoice_date) >= ADD_MONTHS(lfm.full_month, -1)
      AND DATE_TRUNC('MONTH', fi.invoice_date) <= lfm.full_month
    GROUP BY ds.store_id, ds.store_name, ds.manager_name, DATE_TRUNC('MONTH', fi.invoice_date)
)
SELECT
    c.store_id, c.store_name, c.manager_name,
    c.invoice_month AS reporting_month,
    c.revenue AS current_mtd_revenue, c.completed_orders AS current_mtd_orders,
    COALESCE(p.revenue, 0) AS prev_mtd_revenue, COALESCE(p.completed_orders, 0) AS prev_mtd_orders,
    ROUND(((c.revenue - COALESCE(p.revenue, 0)) / NULLIF(p.revenue, 0)) * 100, 2) AS revenue_growth_pct,
    c.completed_orders - COALESCE(p.completed_orders, 0) AS orders_change
FROM monthly_stats c
LEFT JOIN monthly_stats p
    ON c.store_id = p.store_id
    AND p.invoice_month = ADD_MONTHS(c.invoice_month, -1)
WHERE c.invoice_month = (SELECT full_month FROM latest_full_month);

SELECT * FROM workspace.reports.kpi_1_mtd_performance ORDER BY current_mtd_revenue DESC

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_2_avg_days_in_shop AS
SELECT
    ds.store_id, ds.store_name, fo.service_type,
    COUNT(*) AS total_orders,
    ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0), 2) AS avg_days_in_shop,
    ROUND(MIN(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0), 2) AS min_days,
    ROUND(MAX(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0), 2) AS max_days
FROM workspace.cube.fact_order fo
JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
WHERE fo.order_status = 'COMPLETED'
  AND fo.vehicle_out_datetime IS NOT NULL
  AND fo.vehicle_in_datetime IS NOT NULL
GROUP BY ds.store_id, ds.store_name, fo.service_type;

SELECT * FROM workspace.reports.kpi_2_avg_days_in_shop ORDER BY avg_days_in_shop DESC

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_3_survey_coverage AS
SELECT
    ds.store_id, ds.store_name, ds.manager_name,
    COUNT(*) AS surveys_sent,
    SUM(CASE WHEN fs.responded_flag = TRUE THEN 1 ELSE 0 END) AS surveys_responded,
    SUM(CASE WHEN fs.responded_flag = FALSE THEN 1 ELSE 0 END) AS surveys_not_responded,
    ROUND(SUM(CASE WHEN fs.responded_flag = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS response_rate_pct
FROM workspace.cube.fact_survey fs
JOIN workspace.cube.fact_order fo ON fs.order_id = fo.order_id
JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
GROUP BY ds.store_id, ds.store_name, ds.manager_name;

SELECT * FROM workspace.reports.kpi_3_survey_coverage ORDER BY response_rate_pct DESC

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_4_survey_scores AS
SELECT
    ds.store_id, ds.store_name,
    COUNT(*) AS total_responses,
    ROUND(AVG(fs.delivered_on_time_rating), 2) AS avg_on_time_rating,
    ROUND(AVG(fs.work_quality_rating), 2) AS avg_work_quality,
    ROUND(AVG(fs.cleanliness_rating), 2) AS avg_cleanliness,
    ROUND(AVG(fs.communication_rating), 2) AS avg_communication,
    ROUND(AVG(fs.overall_satisfaction_rating), 2) AS avg_overall_satisfaction,
    RANK() OVER (ORDER BY AVG(fs.overall_satisfaction_rating) DESC) AS satisfaction_rank
FROM workspace.cube.fact_survey fs
JOIN workspace.cube.fact_order fo ON fs.order_id = fo.order_id
JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
WHERE fs.responded_flag = TRUE
GROUP BY ds.store_id, ds.store_name;

SELECT * FROM workspace.reports.kpi_4_survey_scores ORDER BY satisfaction_rank

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_5_revenue_vs_budget AS
WITH monthly_revenue AS (
    SELECT
        fo.store_id,
        DATE_FORMAT(fi.invoice_date, 'yyyy-MM') AS month,
        SUM(fi.invoice_amount) AS actual_revenue
    FROM workspace.cube.fact_invoice fi
    JOIN workspace.cube.fact_order fo ON fi.order_id = fo.order_id
    GROUP BY fo.store_id, DATE_FORMAT(fi.invoice_date, 'yyyy-MM')
)
SELECT
    ds.manager_name, mr.month,
    SUM(mr.actual_revenue) AS total_revenue,
    SUM(fb.budget_amount) AS total_budget,
    SUM(mr.actual_revenue) - SUM(fb.budget_amount) AS variance,
    ROUND(SUM(mr.actual_revenue) * 100.0 / NULLIF(SUM(fb.budget_amount), 0), 2) AS achievement_pct,
    RANK() OVER (
        PARTITION BY mr.month
        ORDER BY SUM(mr.actual_revenue) * 100.0 / NULLIF(SUM(fb.budget_amount), 0) DESC
    ) AS manager_rank
FROM monthly_revenue mr
JOIN workspace.cube.fact_budget fb
    ON mr.store_id = fb.ns_store_id AND mr.month = fb.month
JOIN workspace.cube.dim_store ds ON mr.store_id = ds.store_id
GROUP BY ds.manager_name, mr.month;

SELECT * FROM workspace.reports.kpi_5_revenue_vs_budget ORDER BY month DESC, achievement_pct DESC

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_6_top_technicians AS
WITH tech_accuracy AS (
    SELECT
        ds.store_id, ds.store_name, dt.technician_id, dt.technician_name,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN fo.actual_completion_datetime <= fo.planned_completion_datetime THEN 1 ELSE 0 END) AS on_time_completions,
        ROUND(
            SUM(CASE WHEN fo.actual_completion_datetime <= fo.planned_completion_datetime THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        ) AS accuracy_pct,
        ROUND(AVG(DATEDIFF(fo.actual_completion_datetime, fo.planned_completion_datetime)), 2) AS avg_days_variance
    FROM workspace.cube.fact_order fo
    JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
    JOIN workspace.cube.dim_technician dt ON fo.technician_id = dt.technician_id
    WHERE fo.actual_completion_datetime IS NOT NULL
      AND fo.planned_completion_datetime IS NOT NULL
    GROUP BY ds.store_id, ds.store_name, dt.technician_id, dt.technician_name
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY accuracy_pct DESC, total_orders DESC) AS rank
    FROM tech_accuracy
)
SELECT store_id, store_name, technician_id, technician_name,
    total_orders, on_time_completions, accuracy_pct, avg_days_variance, rank
FROM ranked
WHERE rank <= 5;

SELECT * FROM workspace.reports.kpi_6_top_technicians ORDER BY store_id, rank

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_7_ytd_revenue_growth AS
WITH data_boundary AS (
    -- Latest full year = year before the max year in data
    SELECT
        (SELECT MAX(YEAR(invoice_date)) - 1 FROM workspace.cube.fact_invoice) AS reporting_year
),
ytd_revenue AS (
    SELECT
        fo.store_id,
        YEAR(fi.invoice_date) AS revenue_year,
        SUM(fi.invoice_amount) AS ytd_revenue
    FROM workspace.cube.fact_invoice fi
    JOIN workspace.cube.fact_order fo ON fi.order_id = fo.order_id
    GROUP BY fo.store_id, YEAR(fi.invoice_date)
)
SELECT
    ds.store_id, ds.store_name, ds.manager_name,
    c.revenue_year AS reporting_year,
    c.ytd_revenue AS current_ytd_revenue,
    COALESCE(p.ytd_revenue, 0) AS prev_ytd_revenue,
    c.ytd_revenue - COALESCE(p.ytd_revenue, 0) AS ytd_growth,
    ROUND(((c.ytd_revenue - COALESCE(p.ytd_revenue, 0)) / NULLIF(p.ytd_revenue, 0)) * 100, 2) AS ytd_growth_pct,
    RANK() OVER (ORDER BY ((c.ytd_revenue - COALESCE(p.ytd_revenue, 0)) / NULLIF(p.ytd_revenue, 0)) DESC) AS growth_rank
FROM ytd_revenue c
JOIN workspace.cube.dim_store ds ON c.store_id = ds.store_id
LEFT JOIN ytd_revenue p
    ON c.store_id = p.store_id
    AND p.revenue_year = c.revenue_year - 1
WHERE c.revenue_year = (SELECT reporting_year FROM data_boundary);

SELECT * FROM workspace.reports.kpi_7_ytd_revenue_growth ORDER BY growth_rank

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_8_stage_cycle_time AS
SELECT
    ds.store_id, ds.store_name, fo.service_type,
    COUNT(*) AS completed_orders,
    ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.actual_work_start_datetime) / 24.0), 2) AS avg_days_intake_to_start,
    ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.actual_work_start_datetime, fo.actual_completion_datetime) / 24.0), 2) AS avg_days_start_to_completion,
    ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.actual_completion_datetime, fo.actual_delivery_datetime) / 24.0), 2) AS avg_days_completion_to_delivery,
    ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.actual_delivery_datetime) / 24.0), 2) AS avg_total_cycle_days
FROM workspace.cube.fact_order fo
JOIN workspace.cube.dim_store ds ON fo.store_id = ds.store_id
WHERE fo.order_status = 'COMPLETED'
  AND fo.actual_work_start_datetime IS NOT NULL
  AND fo.actual_completion_datetime IS NOT NULL
  AND fo.actual_delivery_datetime IS NOT NULL
GROUP BY ds.store_id, ds.store_name, fo.service_type;

SELECT * FROM workspace.reports.kpi_8_stage_cycle_time ORDER BY store_id, service_type

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_9_estimator_accuracy AS
WITH estimate_versions AS (
    SELECT
        fe.order_id, fe.estimator_id,
        MIN(CASE WHEN fe.version_no = 1 THEN fe.estimate_amount END) AS initial_estimate,
        MAX_BY(fe.estimate_amount, fe.version_no) AS final_estimate
    FROM workspace.cube.fact_estimate fe
    WHERE fe.estimate_amount IS NOT NULL
    GROUP BY fe.order_id, fe.estimator_id
)
SELECT
    de.estimator_id, de.estimator_name,
    COUNT(*) AS total_estimates,
    ROUND(AVG(ev.initial_estimate), 2) AS avg_initial_estimate,
    ROUND(AVG(ev.final_estimate), 2) AS avg_final_estimate,
    ROUND(AVG(ABS(ev.final_estimate - ev.initial_estimate)), 2) AS avg_absolute_deviation,
    ROUND(
        AVG(CASE WHEN ev.initial_estimate > 0
            THEN ABS(ev.final_estimate - ev.initial_estimate) * 100.0 / ev.initial_estimate
            ELSE NULL END
        ), 2
    ) AS avg_deviation_pct,
    ROUND(
        100 - AVG(CASE WHEN ev.initial_estimate > 0
            THEN ABS(ev.final_estimate - ev.initial_estimate) * 100.0 / ev.initial_estimate
            ELSE NULL END
        ), 2
    ) AS accuracy_pct,
    RANK() OVER (
        ORDER BY AVG(CASE WHEN ev.initial_estimate > 0
            THEN ABS(ev.final_estimate - ev.initial_estimate) * 100.0 / ev.initial_estimate
            ELSE NULL END) ASC
    ) AS accuracy_rank
FROM estimate_versions ev
JOIN workspace.cube.dim_estimator de ON ev.estimator_id = de.estimator_id
WHERE ev.initial_estimate IS NOT NULL AND ev.final_estimate IS NOT NULL
GROUP BY de.estimator_id, de.estimator_name;

SELECT * FROM workspace.reports.kpi_9_estimator_accuracy ORDER BY accuracy_rank

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.reports.kpi_10_technician_workload AS
WITH tech_monthly AS (
    SELECT
        dt.technician_id, dt.technician_name,
        DATE_FORMAT(fo.vehicle_in_datetime, 'yyyy-MM') AS work_month,
        COUNT(*) AS orders_handled,
        ROUND(SUM(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0), 2) AS total_days_in_shop,
        ROUND(AVG(TIMESTAMPDIFF(HOUR, fo.vehicle_in_datetime, fo.vehicle_out_datetime) / 24.0), 2) AS avg_days_per_order
    FROM workspace.cube.fact_order fo
    JOIN workspace.cube.dim_technician dt ON fo.technician_id = dt.technician_id
    WHERE fo.order_status = 'COMPLETED'
      AND fo.vehicle_in_datetime IS NOT NULL
      AND fo.vehicle_out_datetime IS NOT NULL
    GROUP BY dt.technician_id, dt.technician_name, DATE_FORMAT(fo.vehicle_in_datetime, 'yyyy-MM')
)
SELECT *,
    RANK() OVER (PARTITION BY work_month ORDER BY orders_handled DESC) AS workload_rank
FROM tech_monthly;

SELECT * FROM workspace.reports.kpi_10_technician_workload ORDER BY work_month DESC, workload_rank LIMIT 200